---
title: nn.Module
numbering: false
---

# 0. 개요
`nn,Module`은 학습할 파라미터와 forward 계산을 묶는 하나의 틀이다. 이 틀을 사용하면 PyTorch가 자동으로 파라미터를 찾아서 optimzer에게 넘기고 GPU에 올리며 저장도 해준다. 그러나 등록 규칙을 잘 지켜야만 동작한다.
+ 규칙을 기억하자


# 1. 기본적인 형태

In [1]:
import torch
from torch import nn

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden) # layer 는 __init__ 에서 정의한다.
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x): # 계산은 forward 함수에
        return self.fc2(self.act(self.fc1(x)))

model = MLP(4, 8, 2) # 모델 인스턴스 생성
x = torch.randn(5, 4)
print(model(x).shape) # torch.Size([5, 2])
print(model) # 등록된 하위 모듈의 구조가 출력된다

torch.Size([5, 2])
MLP(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (act): ReLU()
  (fc2): Linear(in_features=8, out_features=2, bias=True)
)


# 2. 파라미터

In [2]:
for name, p in model.named_parameters():
    print(name, tuple(p.shape), p.requires_grad)

fc1.weight (8, 4) True
fc1.bias (8,) True
fc2.weight (2, 8) True
fc2.bias (2,) True


In [3]:
sum(p.numel() for p in model.parameters()) # 전체 파라미터의 수

58

+ 파라미터는 `requires_grad=True` 가 기본
+ weight shape 가 반대인 이유는 내부에서 `x @ weight.T` 로 연산이 일어나기 때문이다.

# 3. 하위 모듈의 등록 규칙

+ `self.xxx = 모듈` 형태로 속성에 직접 대입한 모듈만 자동으로 등록된다. 리스트에만 담으면 안된다.

In [4]:
# 나쁜 예시
class Bad(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(4, 4) for _ in range(3)] # 파이썬 리스트로 주면 안됨

+ 위와 같은 경우에는 forward 계산은 정상적으로 되지만 파라미터 목록에는 보이지 않아 optimizer가 학습시키지 못함. .to(device)도 안되고 저장도 안됨. 에러 없이 학습만 안됨.

In [5]:
# 좋은 예시
class Gooood(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 4) for _ in range(3)]) # 등록 가능

+ 만약에 딕셔너리가 필요한 경우 `nn.ModuleDict` 를 사용하면 된다.

# 4. 버퍼
+ 학습은 안하지만 모델과 함께 다니는 상태

In [6]:
class Normalizer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim)) # 파라미터 : 학습
        self.register_buffer("mean", torch.zeros(dim)) # 버퍼 : 학습 안됨

    def forward(self, x):
        return (x - self.mean) * self.scale

n = Normalizer(3)

print([k for k, _ in n.named_parameters()])
print([k for k, _ in n.named_buffers()])
print(list(n.state_dict().keys()))

['scale']
['mean']
['scale', 'mean']


# 5. train과 eval

In [7]:
drop = nn.Dropout(p=0.5)
x = torch.ones(1, 8)
drop.train()
drop(x) # 무작위로 끄고 남은 값은 1/(1-p) 여기서는 2배가 된다

tensor([[2., 0., 0., 2., 0., 0., 0., 0.]])

In [8]:
drop.eval()
drop(x) # 그대로 통과함

tensor([[1., 1., 1., 1., 1., 1., 1., 1.]])

+ train() 과 eval()은 Dropout 과 BatchNorm 처럼 학습과 평가 때 동작이 다른 층의 스위치임. BatchNorm 은 train 모드에서 배치별 통계를 쓰면서 running 통계를 갱신하고 eval 모드에서는 저장된 running 통계를 사용
+ eval() 은 그래디언트를 끄는 것이 아님

In [9]:
m = nn.Sequential(nn.Linear(8, 2), nn.Dropout(0.5))
m.eval()
m(x).requires_grad # True 가 나온다. computational graph 는 여전히 만들어진다는 뜻

True

+ 따라서 평가 시에는 model.eval()로 동작을 바꾸는 것 + with torch.no_grad()로 compuational graph 도 끈다

# 6. freeze: 사전학습된 부분을 고정시키는 것

In [10]:
class EncHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(4, 8), nn.BatchNorm1d(8), nn.ReLU())
        self.head = nn.Linear(8, 2)
        self._frozen = False

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False # 그래디언트를 받지 않게 설정
        self.encoder.eval() # BatchNorm 버퍼가 갱신되지 않게 설정
        self._frozen = True

    def forward(self, x):
        return self.head(self.encoder(x))

In [11]:
# optimizer 에는 학습할 파라미터만 넘기는 것이 깔끔하다
opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3)